# Decipher Visualization

Visualise the multi-LLM deciphering output of the pipeline in [src/decipher/](../src/decipher/).

**Prerequisite:** run `python scripts/decipher_all_segments.py --mock --only 20231221180251` first (or a real run with API key) so `predictions/decipher/{seg}/result.json` exists.

In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from decipher.model_registry import OPEN_SOURCE_MODELS, MANUAL_MODELS

SEG_ID = '20231221180251'
RESULT = ROOT / 'predictions' / 'decipher' / SEG_ID / 'result.json'
print('result:', RESULT, 'exists:', RESULT.exists())

In [ ]:
data = json.loads(RESULT.read_text(encoding='utf-8'))
print(f"segment       : {data['seg_id']}")
print(f"n_strips      : {data['n_strips']}")
print(f"open-source   : {len(data['models_used_open_source'])}")
for slug in data['models_used_open_source']: print('  -', slug)
print(f"manual        : {len(data['models_used_manual'])}")
for slug in data['models_used_manual']: print('  -', slug)
print(f"mock          : {data.get('mock', False)}")

## 1. Per-strip view — strip image with each model's reading overlaid

In [ ]:
STRIPS_DIR = RESULT.parent / 'strips'

TIER_COLOR = {'HIGH': '#1f9d55', 'MED': '#d68a00', 'LOW': '#b00020'}

def plot_strip(strip):
    img = np.asarray(Image.open(STRIPS_DIR / Path(strip['image_path']).name))
    models = list(strip['per_model'].keys())
    n = len(models)
    fig, axes = plt.subplots(n + 2, 1, figsize=(13, 1.0 + 0.7 * n + 1.5),
                             gridspec_kw={'height_ratios': [1.6] + [0.5] * n + [1.0]})
    axes[0].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title(f"strip {strip['strip_id']:02d}  y={strip['y_range'][0]}..{strip['y_range'][1]}  x={strip['x_range'][0]}..{strip['x_range'][1]}",
                     fontsize=9, loc='left')
    axes[0].axis('off')
    W = img.shape[1]
    for ax, slug in zip(axes[1:1+n], models):
        entry = strip['per_model'][slug]
        parsed = entry.get('parsed') or {}
        chars = parsed.get('characters', [])
        ax.set_xlim(0, W); ax.set_ylim(0, 1); ax.set_facecolor('#f6f5f1')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_ylabel(entry.get('display_name', slug), fontsize=8, rotation=0,
                      ha='right', va='center', labelpad=70)
        for c in chars:
            x = float(c.get('x_norm', 0.5)) * W
            conf = float(c.get('confidence', 0.5))
            color = '#1f9d55' if conf > 0.75 else ('#d68a00' if conf > 0.5 else '#b00020')
            ax.text(x, 0.5, c.get('char', '·'), color=color, fontsize=14,
                    ha='center', va='center', fontweight='bold',
                    family='DejaVu Sans')
        if entry.get('error'):
            ax.text(W / 2, 0.5, f"[error: {entry['error'][:60]}]",
                    ha='center', va='center', color='#b00020', fontsize=8)
    # consensus row
    ax = axes[-1]
    ax.set_xlim(0, W); ax.set_ylim(0, 1); ax.set_facecolor('#1a1a18')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_ylabel('CONSENSUS', fontsize=8, rotation=0, ha='right', va='center',
                  labelpad=70, color='#caa15a')
    for c in strip['consensus']['characters']:
        x = float(c['x_norm']) * W
        ax.text(x, 0.5, c['char'], color=TIER_COLOR.get(c['tier'], '#cccccc'),
                fontsize=18, ha='center', va='center', fontweight='bold',
                family='DejaVu Sans')
    fig.tight_layout()
    return fig

for s in data['strips'][:3]:
    fig = plot_strip(s)
    plt.show()

## 2. Agreement heatmap — which models agreed at which positions

In [ ]:
def agreement_heatmap(strip):
    chars = strip['consensus']['characters']
    if not chars: return None
    models = list(strip['per_model'].keys())
    M = np.zeros((len(models), len(chars)))
    for j, c in enumerate(chars):
        for v in c['votes']:
            if v['char'] == c['char']:
                try:
                    i = models.index(v['model'])
                    M[i, j] = 1.0
                except ValueError:
                    pass
    fig, ax = plt.subplots(figsize=(0.6 * len(chars) + 2, 0.4 * len(models) + 1))
    ax.imshow(M, aspect='auto', cmap='Greens', vmin=0, vmax=1)
    ax.set_xticks(range(len(chars))); ax.set_xticklabels([c['char'] for c in chars])
    ax.set_yticks(range(len(models)))
    ax.set_yticklabels([strip['per_model'][m].get('display_name', m) for m in models],
                       fontsize=8)
    ax.set_title(f"strip {strip['strip_id']:02d}: agreement with consensus", fontsize=9)
    return fig

for s in data['strips'][:3]:
    fig = agreement_heatmap(s)
    if fig: plt.show()

## 3. Aggregate — consensus text per strip + overall tier counts

In [ ]:
tier_counts = {'HIGH': 0, 'MED': 0, 'LOW': 0}
print(f"{'strip':>6}  {'consensus':<32}  HIGH MED LOW")
print('-' * 60)
for s in data['strips']:
    h = m = l = 0
    for c in s['consensus']['characters']:
        if c['tier'] == 'HIGH': h += 1
        elif c['tier'] == 'MED': m += 1
        else: l += 1
    tier_counts['HIGH'] += h; tier_counts['MED'] += m; tier_counts['LOW'] += l
    print(f"  {s['strip_id']:02d}   {s['consensus']['text'][:32]:<32}  {h:4d} {m:3d} {l:3d}")
print('-' * 60)
total = sum(tier_counts.values()) or 1
print(f"  totals  {'':<32}  {tier_counts['HIGH']:4d} {tier_counts['MED']:3d} {tier_counts['LOW']:3d}")
print(f"           HIGH {tier_counts['HIGH']/total:.1%} | MED {tier_counts['MED']/total:.1%} | LOW {tier_counts['LOW']/total:.1%}")